# PySpark

## Sistema de recomendación de productos

El objetivo de este ejercicio es construir un sistema de
recomendación utilizando **PySpark**.

El sistema recibe un producto como parámetro mediante la
función `getRelatedProducts` y busca productos relacionados
considerando:

- Categoría.
- Palabras similares en el título.
- Cercanía de precio.

El dataset utilizado se encuentra en el archivo
`productos.csv` y fue construido a partir del scraper del
ejercicio anterior utilizando Books to Scrape.


## 1. Dataset de productos

El archivo `productos.csv` contiene 20 productos y los
siguientes campos:

- `id`
- `titulo`
- `categoria`
- `precio`
- `rating`
- `url`

Los productos fueron obtenidos mediante Web Scraping desde:

https://books.toscrape.com/


## 2. Crear la sesión de PySpark

Se utiliza `SparkSession` para trabajar con el dataset
mediante DataFrames de PySpark.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SistemaRecomendacionProductos")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

## 3. Cargar el CSV

PySpark lee directamente el archivo `productos.csv`. Se
indica que la primera fila contiene los encabezados y que
Spark debe inferir automáticamente los tipos de datos.


In [ ]:
productos = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("productos.csv")
)

productos.select(
    "id",
    "titulo",
    "categoria",
    "precio",
).show(
    20,
    truncate=False,
)

## 4. Función getRelatedProducts

La función recibe como parámetro el ID del producto que el
usuario está observando.

Posteriormente compara ese producto contra el resto del
catálogo y asigna una puntuación.

### Sistema de puntuación

**Categoría**

- Misma categoría: 5 puntos.

**Título**

- Si comparte alguna palabra relevante del título:
  3 puntos.

**Precio**

- Diferencia de hasta £5: 2 puntos.
- Diferencia de hasta £10: 1 punto.

Finalmente se ordenan los productos por puntuación y se
devuelven los cinco productos relacionados mejor posicionados.


In [ ]:
def getRelatedProducts(producto):
    producto_actual = (
        productos
        .filter(F.col("id") == producto)
        .first()
    )

    if producto_actual is None:
        print(
            f"No existe un producto con ID {producto}"
        )
        return None

    id_actual = producto_actual["id"]
    titulo_actual = producto_actual["titulo"]
    categoria_actual = producto_actual["categoria"]
    precio_actual = producto_actual["precio"]

    palabras_titulo = [
        palabra.lower()
        for palabra in titulo_actual.split()
        if len(palabra) > 3
    ]

    relacionados = productos.filter(
        F.col("id") != id_actual
    )

    relacionados = relacionados.withColumn(
        "puntos_categoria",
        F.when(
            F.col("categoria") == categoria_actual,
            5,
        ).otherwise(0),
    )

    condicion_palabras = F.lit(False)

    for palabra in palabras_titulo:
        condicion_palabras = (
            condicion_palabras
            | F.lower(
                F.col("titulo")
            ).contains(palabra)
        )

    relacionados = relacionados.withColumn(
        "puntos_titulo",
        F.when(
            condicion_palabras,
            3,
        ).otherwise(0),
    )

    diferencia_precio = F.abs(
        F.col("precio") - F.lit(precio_actual)
    )

    relacionados = relacionados.withColumn(
        "puntos_precio",
        F.when(
            diferencia_precio <= 5,
            2,
        ).when(
            diferencia_precio <= 10,
            1,
        ).otherwise(0),
    )

    relacionados = relacionados.withColumn(
        "puntuacion",
        F.col("puntos_categoria")
        + F.col("puntos_titulo")
        + F.col("puntos_precio"),
    )

    relacionados = (
        relacionados
        .filter(
            F.col("puntuacion") > 0
        )
        .orderBy(
            F.desc("puntuacion"),
            F.asc(diferencia_precio),
        )
        .limit(5)
    )

    print("\nProducto seleccionado:")
    print(
        f"ID: {id_actual}"
    )
    print(
        f"Título: {titulo_actual}"
    )
    print(
        f"Categoría: {categoria_actual}"
    )
    print(
        f"Precio: £{precio_actual}"
    )

    print(
        "\nProductos relacionados:"
    )

    relacionados.select(
        "id",
        "titulo",
        "categoria",
        "precio",
        "puntuacion",
    ).show(
        truncate=False
    )

    return relacionados

## 5. Prueba 1: producto de categoría Poetry

Se simula que el usuario está observando el producto con
ID `1`:

**A Light in the Attic**

- Categoría: Poetry
- Precio: £51.77

Se ejecuta:


In [ ]:
getRelatedProducts(1)

### Resultado de la prueba 1

Los principales productos relacionados obtenidos fueron:

| Producto | Categoría | Precio | Puntuación |
|---|---|---:|---:|
| The Black Maria | Poetry | £52.15 | 7 |
| Olio | Poetry | £23.88 | 5 |
| Shakespeare's Sonnets | Poetry | £20.66 | 5 |
| Libertarianism for Beginners | Politics | £51.33 | 2 |
| Scott Pilgrim's Precious Little Life | Sequential Art | £52.29 | 2 |

**The Black Maria** obtuvo la puntuación más alta porque
pertenece a la misma categoría y además tiene un precio muy
cercano al producto seleccionado.


## 6. Prueba 2: producto de categoría Music

También se realizó una segunda prueba con el producto
ID `16`:

**Our Band Could Be Your Life: Scenes from the American Indie
Underground, 1981-1991**

- Categoría: Music
- Precio: £57.25


In [ ]:
getRelatedProducts(16)

### Resultado de la prueba 2

Los principales productos relacionados fueron:

| Producto | Categoría | Precio | Puntuación |
|---|---|---:|---:|
| Rip it Up and Start Again | Music | £35.02 | 5 |
| The Dirty Little Secrets of Getting Your Dream Job | Business | £33.34 | 3 |
| The Boys in the Boat | Default | £22.60 | 3 |
| Sapiens: A Brief History of Humankind | History | £54.23 | 2 |
| Tipping the Velvet | Historical Fiction | £53.74 | 2 |

**Rip it Up and Start Again** apareció en primer lugar porque
comparte la categoría `Music` con el producto seleccionado.


## Conclusión

Se construyó un sistema básico de recomendación de productos
utilizando PySpark.

La función `getRelatedProducts` recibe un producto como
parámetro y utiliza operaciones sobre DataFrames para comparar
categoría, palabras del título y precio.

Las pruebas realizadas con productos de distintas categorías
demostraron que el sistema modifica sus recomendaciones en
función del producto seleccionado.


In [ ]:
spark.stop()